# SceneTwin — Single Brain Render Notebook

Use this notebook when your Colab runtime reset but you still have a saved TRIBE prediction file such as `video_preds.npy` or `desc_accurate_detailed_preds.npy`.

It uploads a `.npy` prediction array and renders a cleaner single brain view instead of the split multi-view brain image.

## 1. Install TRIBE plotting dependencies

Run this first. If Colab asks you to restart after installs, restart and run this cell again.

In [ ]:
!pip install -q git+https://github.com/facebookresearch/tribev2.git
!pip install -q nilearn nibabel matplotlib numpy

## 2. Upload a prediction file

Upload a file like:

- `video_preds.npy`
- `desc_accurate_detailed_preds.npy`
- any TRIBE prediction array with shape `(timepoints, 20484)` or `(20484,)`

In [ ]:
from google.colab import files
uploaded = files.upload()

pred_path = next(iter(uploaded.keys()))
print('Uploaded:', pred_path)

## 3. Load and inspect predictions

In [ ]:
import numpy as np

preds = np.load(pred_path)
print('Prediction shape:', preds.shape)

if preds.ndim == 2:
    brain = preds.mean(axis=0)
elif preds.ndim == 1:
    brain = preds
else:
    raise ValueError(f'Expected 1D or 2D array, got shape {preds.shape}')

print('Brain vector shape:', brain.shape)
assert brain.shape[0] == 20484, 'Expected 20484 fsaverage5 vertices from TRIBE v2.'

## 4. Render one clean brain view

Recommended views:

- `dorsal`: top-down, best single poster view
- `posterior`: back view
- `anterior`: front view
- `left`: left hemisphere side
- `right`: right hemisphere side

In [ ]:
import matplotlib.pyplot as plt
from tribev2.plotting import PlotBrainNilearn

# Change this if you want another view.
VIEW = 'dorsal'

# Good names: video, accurate_detailed, hallucinated, etc.
LABEL = pred_path.replace('.npy', '')

plotter = PlotBrainNilearn(
    mesh='fsaverage5',
    inflate='half',
    hemisphere_gap=0,
)

fig, ax = plt.subplots(
    1, 1,
    figsize=(6, 6),
    subplot_kw={'projection': '3d'}
)

plotter.plot_surf(
    brain,
    views=[VIEW],
    axes=[ax],
    cmap='RdBu_r',
    symmetric_cbar=True,
    colorbar=False,
)

ax.set_title(f'TRIBE predicted cortical response — {LABEL}', fontsize=12)
out_path = f'/content/brain_single_{LABEL}_{VIEW}.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('Saved:', out_path)

## 5. Download the rendered brain image

In [ ]:
from google.colab import files
files.download(out_path)

## Optional: Render several views at once

Run this if you want a small set of clean poster candidates.

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

views = ['dorsal', 'posterior', 'anterior']
paths = []

for view in views:
    fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw={'projection': '3d'})
    plotter.plot_surf(
        brain,
        views=[view],
        axes=[ax],
        cmap='RdBu_r',
        symmetric_cbar=True,
        colorbar=False,
    )
    ax.set_title(f'{LABEL} — {view}', fontsize=12)
    path = f'/content/brain_single_{LABEL}_{view}.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    paths.append(path)

for path in paths:
    files.download(path)